# N163 · JavaScript异步、定时器与并发限制

**目标：** 分离Promise结果竞速和底层任务取消。

**先修：** N161, N162, N053。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Promise/async/await；微任务；超时；debounce/throttle；并发池；异常传播。

## 从问题到算法

并发结果的顺序与完成时间无关：启动时分配索引，完成时填入对应位置。Promise 的一次拒绝不会自动取消已经开始的其他任务。

超时包装只改变调用者何时得到拒绝；底层函数仍可能运行。成功、失败两条路径都清理计时器。debounce 每次取消上一个待执行回调，最后一次参数与 this 才被使用。本章允许注入时钟以确定性测试边界，并保留真实 Node 定时器测试。

并发池只有 n 个 worker，每个 worker 同时等待一个任务。遇到首个错误停止领新任务，等已经开始的任务收尾后传播首个错误。本教学接口返回按输入排序的结果；要求仅完成信号的平台接口可忽略该返回数组。

## 最小实现

**本章接口：** `promiseAll(functions)`、`timeLimit(fn, t)`、`debounce(fn, t)`、`promisePool(functions, n)`

In [1]:
JS_SOURCE = r''''use strict';
function promiseAll(functions) {
  return new Promise((resolve,reject)=>{
    const out=new Array(functions.length);
    if (functions.length===0) { resolve(out);return; }
    let remaining=functions.length;
    functions.forEach((fn,i)=>{
      Promise.resolve().then(fn).then(value=>{
        out[i]=value;
        if (--remaining===0) resolve(out);
      },reject);
    });
  });
}
function timeLimit(fn,t,clock=globalThis) {
  return function(...args) {
    const receiver=this;
    return new Promise((resolve,reject)=>{
      const timer=clock.setTimeout(()=>reject('Time Limit Exceeded'),t);
      Promise.resolve().then(()=>fn.apply(receiver,args)).then(
        value=>{clock.clearTimeout(timer);resolve(value);},
        error=>{clock.clearTimeout(timer);reject(error);});
    });
  };
}
function debounce(fn,t,clock=globalThis) {
  let timer=null;
  function debounced(...args) {
    if (timer!==null) clock.clearTimeout(timer);
    const receiver=this;
    timer=clock.setTimeout(()=>{timer=null;fn.apply(receiver,args);},t);
  }
  debounced.cancel=()=>{
    if (timer!==null) clock.clearTimeout(timer);
    timer=null;
  };
  return debounced;
}
async function promisePool(functions,n) {
  if (!Number.isInteger(n)||n<1) throw new RangeError('Positive concurrency required');
  const out=new Array(functions.length);
  let next=0,failed=false,firstError;
  async function worker() {
    while (!failed && next<functions.length) {
      const index=next++;
      try { out[index]=await functions[index](); }
      catch (error) { if (!failed) firstError=error; failed=true; }
    }
  }
  await promiseAll(Array.from({length:Math.min(n,functions.length)},()=>worker));
  if (failed) throw firstError;
  return out;
}
'''

import json
import subprocess

def run_js(extra):
    result = subprocess.run(['node', '-e', JS_SOURCE + '\n' + extra],
                            text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 正确性、前提与复杂度

worker 取得索引与递增 next 之间没有 await，因此不会领到重复任务；一个 worker 完成当前任务后才领下一个，同时活跃任务数至多 n。debounce 的待执行定时器至多一个。超时拒绝不是资源取消，必须分别讨论底层任务的收尾。

**代价：** 忽略任务本身成本，promiseAll 和池调度 O(m)，结果存储 O(m)，池的活跃 worker O(min(n,m))。debounce 每次调用的附加状态 O(1)。定时器测试不以脆弱的墙钟毫秒阈值证明性能。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from IPython.display import Code
display(Code(JS_SOURCE,language='javascript'))
trace = json.loads(run_js("""(async()=>{
const trace=[];let active=0;
const jobs=[3,1,2].map((ms,i)=>async()=>{trace.push(['start',i,++active]);await new Promise(r=>setTimeout(r,ms));trace.push(['finish',i,--active]);return i;});
const result=await promisePool(jobs,2);trace.push(['result',JSON.stringify(result),active]);
console.log(JSON.stringify(trace));
})().catch(e=>{console.error(e);process.exitCode=1;});"""))
show_table(['event','task/result','active'],trace)

'use strict';
function promiseAll(functions) {
  return new Promise((resolve,reject)=>{
    const out=new Array(functions.length);
    if (functions.length===0) { resolve(out);return; }
    let remaining=functions.length;
    functions.forEach((fn,i)=>{
      Promise.resolve().then(fn).then(value=>{
        out[i]=value;
        if (--remaining===0) resolve(out);
      },reject);
    });
  });
}
function timeLimit(fn,t,clock=globalThis) {
  return function(...args) {
    const receiver=this;
    return new Promise((resolve,reject)=>{
      const timer=clock.setTimeout(()=>reject('Time Limit Exceeded'),t);
      Promise.resolve().then(()=>fn.apply(receiver,args)).then(
        value=>{clock.clearTimeout(timer);resolve(value);},
        error=>{clock.clearTimeout(timer);reject(error);});
    });
  };
}
function debounce(fn,t,clock=globalThis) {
  let timer=null;
  function debounced(...args) {
    if (timer!==null) clock.clearTimeout(timer);
    const receiver=this;
    timer=clock.setTimeout(()=>{timer=null;fn.apply(receiver,args);},t);
  }
  debounced.cancel=()=>{
    if (timer!==null) clock.clearTimeout(timer);
    timer=null;
  };
  return debounced;
}
async function promisePool(functions,n) {
  if (!Number.isInteger(n)||n<1) throw new RangeError('Positive concurrency required');
  const out=new Array(functions.length);
  let next=0,failed=false,firstError;
  async function worker() {
    while (!failed && next<functions.length) {
      const index=next++;
      try { out[index]=await functions[index](); }
      catch (error) { if (!failed) firstError=error; failed=true; }
    }
  }
  await promiseAll(Array.from({length:Math.min(n,functions.length)},()=>worker));
  if (failed) throw firstError;
  return out;
}

event,task/result,active
start,0,1
start,1,2
finish,1,1
start,2,2
finish,0,1
finish,2,0
result,"[0,1,2]",0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
print(run_js("""const assert=require('node:assert/strict');
function fakeClock() {
  let now=0,next=0;const tasks=new Map();
  return {
    setTimeout(fn,delay){const id=next++;tasks.set(id,{time:now+delay,fn});return id;},
    clearTimeout(id){tasks.delete(id);},
    advance(delta){const target=now+delta;while(true){
      const ready=[...tasks].filter(([id,x])=>x.time<=target).sort((a,b)=>a[1].time-b[1].time||a[0]-b[0]);
      if(!ready.length)break;const [id,x]=ready[0];tasks.delete(id);now=x.time;x.fn();
    }now=target;},
    size(){return tasks.size;}
  };
}
(async()=>{
  const clock=fakeClock(),seen=[];
  const deb=debounce(function(x){seen.push(this.prefix+x);},10,clock);
  deb.call({prefix:'a'},1);clock.advance(9);assert.deepEqual(seen,[]);
  deb.call({prefix:'b'},2);clock.advance(9);assert.deepEqual(seen,[]);
  clock.advance(1);assert.deepEqual(seen,['b2']);assert.equal(clock.size(),0);
  deb(3);deb.cancel();clock.advance(20);assert.deepEqual(seen,['b2']);

  const originalSet=globalThis.setTimeout,originalClear=globalThis.clearTimeout;
  const activeTimers=new Set();
  globalThis.setTimeout=(fn,delay,...args)=>{
    let timer=originalSet(()=>{activeTimers.delete(timer);fn(...args);},delay);
    activeTimers.add(timer);return timer;
  };
  globalThis.clearTimeout=timer=>{activeTimers.delete(timer);originalClear(timer);};
  const delay=ms=>new Promise(r=>setTimeout(r,ms));
  try {
    assert.deepEqual(await promiseAll([]),[]);
    assert.deepEqual(await promiseAll([async()=>{await delay(5);return 1;},async()=>2]),[1,2]);
    const boom=new Error('boom');
    await assert.rejects(promiseAll([()=>{throw boom;}]),e=>e===boom);
    assert.equal(await timeLimit(async x=>x+1,100)(2),3);
    await assert.rejects(timeLimit(async()=>{throw boom;},100)(),e=>e===boom);
    let finish;
    const underlyingFinished=new Promise(r=>{finish=r;});
    const slow=async()=>{await delay(15);finish();return 9;};
    await assert.rejects(timeLimit(slow,1)(),e=>e==='Time Limit Exceeded');
    await underlyingFinished;
    // Allow the wrapper's completion handler to clear its already-fired timer.
    await Promise.resolve();
    let active=0,maximum=0;
    const jobs=Array.from({length:12},(_,i)=>async()=>{
      active++;maximum=Math.max(maximum,active);
      try {await delay(i%3);return i*i;} finally {active--;}
    });
    assert.deepEqual(await promisePool(jobs,3),Array.from({length:12},(_,i)=>i*i));
    assert(maximum<=3);assert.equal(active,0);
    let starts=0;
    await assert.rejects(promisePool([
      async()=>{starts++;throw boom;},
      async()=>{starts++;await delay(2);return 1;},
      async()=>{starts++;return 2;}
    ],2),e=>e===boom);
    assert.equal(starts,2);
    assert.deepEqual(await promisePool([],2),[]);
    await assert.rejects(promisePool([],0),RangeError);
    let resolveDebounced;
    const done=new Promise(r=>{resolveDebounced=r;});
    const values=[];const realDeb=debounce(x=>{values.push(x);resolveDebounced();},1);
    realDeb(1);realDeb(2);realDeb(3);await done;
    assert.deepEqual(values,[3]);
    assert.equal(activeTimers.size,0);
  } finally {
    for(const timer of activeTimers)originalClear(timer);
    globalThis.setTimeout=originalSet;globalThis.clearTimeout=originalClear;
  }
  console.log('Node async assertions passed');
})().catch(error=>{console.error(error);process.exitCode=1;});"""))
print('N163: 所有本章断言通过')

Node async assertions passed

N163: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释超时拒绝不等于取消原任务。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 用可控时钟测试debounce边界而非脆弱毫秒断言。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 2721. Execute Asynchronous Functions in Parallel（canonical）
- 2637. Promise Time Limit（canonical）：[官方入口](https://leetcode.com/problems/promise-time-limit/)
- 2627. Debounce（canonical）
- 2636. Promise Pool（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。